# Lab 15 Regression and incident drill

**Compute:** CPU; no downloads. **Goal:** Reject an incompatible candidate, rehearse recovery and estimate scaling limits.

Read the matching lecture weeks and lab guide. Run cells in order in a fresh kernel. This is the worked tutorial; complete the independent experiment/report afterward. Save actual results in runs. Outputs are intentionally not pre-filled. Install/download cells need internet and may require a runtime restart. The companion script contains the same Python cells.


In [ ]:
from pathlib import Path
import sys,json,time,random,hashlib,math
if hasattr(sys.stdout,'reconfigure'): sys.stdout.reconfigure(encoding='utf-8')
candidates=[Path.cwd(),Path.cwd().parent,Path.cwd()/'llm_course',Path('/content/llm_course')]
COURSE=next((p for p in candidates if (p/'coursekit'/'core.py').exists()),None)
if COURSE is None:
    raise RuntimeError('Extract the course ZIP first; run from llm_course or its notebooks directory.')
if str(COURSE) not in sys.path: sys.path.insert(0,str(COURSE))
RUNS=COURSE/'runs'
RUNS.mkdir(exist_ok=True)
from coursekit.core import *
random.seed(7)
print('Course root:',COURSE)


In [ ]:
import subprocess
subprocess.check_call([sys.executable,'-m','unittest','discover','-s','tests','-v'],cwd=COURSE)
previous=make_manifest({'release':'qualifying-A'})
candidate=dict(previous);candidate['release']='candidate-B';candidate['corpus_sha256']='incompatible'
try:
    verify_manifest(candidate);decision='eligible_for_further_quality_checks'
except ValueError: decision='reject_and_restore_previous'
assert decision=='reject_and_restore_previous' and verify_manifest(previous)
print(decision)
for scalable in [.7,.8]: print('Ideal 4-way speedup',scalable,1/((1-scalable)+scalable/4))
report={'trigger':'incompatible corpus hash','impact':'candidate refused before release',
        'mitigation':'restore prior manifest','regression':'manifest rejects changed corpus hash',
        'remaining_checks':['semantic quality','real index/model compatibility','load behavior']}
(RUNS/'lab15_incident.json').write_text(json.dumps(report,indent=2))


The simple drill restores metadata, not a hosted production release. Extend to real model/index/prompt artifacts and smoke requests. Add a regression for an actual project failure and write a capacity memo using measured one-device data. The scaling values above are theoretical illustrations.
